<a href="https://colab.research.google.com/github/doucettematthew07-tech/AISmealHTMLTests/blob/main/MLB_Project3_RAG_Chatbot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLB Project 3 - RAG Chatbot

## Project Overview

Welcome to Project 3! In this project, you'll build a **Retrieval-Augmented Generation (RAG)** chatbot that can answer questions about a PDF document.

### What is RAG?
RAG combines two powerful concepts:
1. **Retrieval**: Finding relevant information from a document
2. **Generation**: Using an LLM to generate natural language answers

### What You'll Learn
- How to extract and process text from PDFs
- How to create text embeddings (vector representations)
- How to build a simple vector database
- How to search for relevant information using similarity
- How to use an LLM to generate answers based on context

### Project Structure
1. Setup and imports
2. Build a Vector Database
3. PDF processing utilities
4. Question answering system
5. Put it all together!

---

In [ ]:
# Install required packages (run this cell first!)
!pip install sentence-transformers pypdf transformers huggingface_hub torch tqdm -q

In [ ]:
# Import all necessary libraries
import os
import numpy as np
import warnings
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from transformers import pipeline
from huggingface_hub import login

# Suppress unnecessary warnings for cleaner output
warnings.filterwarnings("ignore", category=UserWarning)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [ ]:
# Configuration settings
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # The language model for generating answers
EMBEDDING_MODEL = "all-MiniLM-L12-v2"  # The model for creating embeddings
HF_API_KEY = os.getenv("HF_API_KEY", "YOUR-HF-API-KEY-HERE")  # Optional Hugging Face API key

print(f"📋 LLM Model: {LLM_MODEL}")
print(f"📋 Embedding Model: {EMBEDDING_MODEL}")

📋 LLM Model: Qwen/Qwen2.5-0.5B-Instruct
📋 Embedding Model: all-MiniLM-L12-v2


In [ ]:
class VectorDB:
    def __init__(self, model_name: str):
        """
        Initialize the Vector Database with an embedding model.

        Args:
            model_name: Name of the SentenceTransformer model to use
        """
        # Load the SentenceTransformer embedding model
        self.embedModel = SentenceTransformer(model_name)

        # Get the embedding dimension from the model
        self.embed_size = self.embedModel.get_sentence_embedding_dimension()

        # Initialize an empty NumPy array for embeddings with the correct shape
        self._embeddings = np.empty((0, self.embed_size))

        # Initialize an empty list for storing the original text strings
        self._strings = []

        print(f"✅ VectorDB initialized with embedding dimension: {self.embed_size}")

In [ ]:
    def addToDatabase(self, input: list[str]):
        """
        Add text chunks to the vector database.

        Args:
            input: List of text strings to add to the database
        """
        # Convert the input strings to embeddings
        new_embeddings = self.embedModel.encode(input)

        # Stack the new embeddings with existing ones
        if self._embeddings.shape[0] == 0:
            self._embeddings = new_embeddings
        else:
            self._embeddings = np.vstack([self._embeddings, new_embeddings])

        # Extend the _strings list with the new input strings
        self._strings.extend(input)

        print(f"✅ Added {len(input)} chunks. Total chunks: {len(self._strings)}")

# Add this method to the VectorDB class
VectorDB.addToDatabase = addToDatabase

In [ ]:
    def clearDatabase(self):
        """
        Clear all data from the vector database.
        """
        # Reset _embeddings to an empty array with the correct shape
        self._embeddings = np.empty((0, self.embed_size))

        # Reset _strings to an empty list
        self._strings = []

        print("🗑️ Database cleared!")

# Add this method to the VectorDB class
VectorDB.clearDatabase = clearDatabase

In [ ]:
    def euclideanSim(self, x, y):
        """
        Calculate Euclidean similarity between two vectors.

        Args:
            x: First vector (numpy array)
            y: Second vector (numpy array)

        Returns:
            Similarity score (higher = more similar)
        """
        # Calculate Euclidean distance using np.linalg.norm()
        distance = np.linalg.norm(x - y)

        # Convert distance to similarity
        similarity = 1 / (1 + distance)

        return similarity

# Add this method to the VectorDB class
VectorDB.euclideanSim = euclideanSim

In [ ]:
    def search(self, query: str, n_return=3):
        """
        Search the database for the most similar chunks to the query.

        Args:
            query: The search query string
            n_return: Number of top results to return

        Returns:
            Tuple of (text_chunks, similarity_scores)
        """
        # Generate an embedding for the query
        query_embedding = self.embedModel.encode([query])[0]

        # Calculate similarity between query and all stored embeddings
        similarities = [self.euclideanSim(query_embedding, emb) for emb in self._embeddings]

        # Find indices of the top n_return most similar results
        # np.argsort returns indices in ascending order, so we take the last n and reverse them
        top_indices = np.argsort(similarities)[-n_return:][::-1]

        # Get the corresponding text chunks and scores
        top_chunks = [self._strings[i] for i in top_indices]
        top_scores = [similarities[i] for i in top_indices]

        return top_chunks, top_scores

# Add this method to the VectorDB class
VectorDB.search = search

In [ ]:
# Test the VectorDB
print("Testing VectorDB...\n")

# Create a test database
test_vdb = VectorDB(EMBEDDING_MODEL)

# Add some test data
test_data = [
    "Python is a programming language.",
    "Machine learning involves training models on data.",
    "Dogs are loyal pets.",
    "Neural networks are inspired by the human brain."
]

test_vdb.addToDatabase(test_data)

# Search for something
query = "What is ML?"
results, scores = test_vdb.search(query, n_return=2)

print(f"\n🔍 Query: '{query}'\n")
for i, (chunk, score) in enumerate(zip(results, scores), 1):
    print(f"Result {i} (similarity: {score:.4f}):")
    print(f"  {chunk}\n")

Testing VectorDB...



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_1101/3447878263.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embed_size = self.embedModel.get_sentence_embedding_dimension()


✅ VectorDB initialized with embedding dimension: 384
✅ Added 4 chunks. Total chunks: 4

🔍 Query: 'What is ML?'

Result 1 (similarity: 0.4539):
  Machine learning involves training models on data.

Result 2 (similarity: 0.4379):
  Neural networks are inspired by the human brain.



In [ ]:
def clean_text(text: str) -> str:
    """
    Clean text by removing extra whitespace and newlines.

    Args:
        text: Raw text string

    Returns:
        Cleaned text string
    """
    # Split text into words and join with single spaces
    return " ".join(text.split())

# Test the function
test_text = "This   has    extra\n\nspaces   and\nnewlines."
print(f"Original: {repr(test_text)}")
print(f"Cleaned:  {repr(clean_text(test_text))}")

Original: 'This   has    extra\n\nspaces   and\nnewlines.'
Cleaned:  'This has extra spaces and newlines.'


In [ ]:
def chunksFromText(text: str, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks.

    Args:
        text: Input text string
        chunk_size: Size of each chunk in characters
        overlap: Number of overlapping characters between chunks

    Returns:
        List of text chunks
    """
    chunks = []

    # Calculate the step size (how much to move forward each time)
    step = chunk_size - overlap

    # Ensure we don't loop forever if step is 0 or less
    if step <= 0:
        return [text]

    # Loop through the text, creating chunks
    for i in range(0, len(text), step):
        # Create a chunk from text[i:i+chunk_size]
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)

        # Stop if we've reached the end of the text
        if i + chunk_size >= len(text):
            break

    return chunks

# Test the function
test_text = "A" * 100  # 100 characters
test_chunks = chunksFromText(test_text, chunk_size=30, overlap=5)
print(f"Text length: {len(test_text)}")
print(f"Number of chunks: {len(test_chunks)}")
print(f"First chunk length: {len(test_chunks[0]) if test_chunks else 0}")

Text length: 100
Number of chunks: 4
First chunk length: 30


In [ ]:
def chunksFromPDF(vDB, path: str, startPage=0, endPage=None):
    """
    Extract text from a PDF, chunk it, and add to the vector database.

    Args:
        vDB: VectorDB instance to add chunks to
        path: Path to the PDF file
        startPage: First page to process (0-indexed)
        endPage: Last page to process (None = all pages)
    """
    # Create a PdfReader object
    reader = PdfReader(path)

    # Get the list of pages to process
    pages = reader.pages[startPage:endPage]

    print(f"📄 Processing {len(pages)} pages from PDF...")

    all_chunks = []

    # Loop through each page with tqdm for progress bar
    for page_num, page in enumerate(tqdm(pages, desc="Extracting text"), startPage):
        # Extract text from the page
        text = page.extract_text()

        # Clean the text using the clean_text function
        text = clean_text(text)

        # Skip empty or very short pages (likely covers or blank pages)
        if len(text) < 100:
            continue

        # Convert text to chunks using chunksFromText
        page_chunks = chunksFromText(text)

        # Add page number to each chunk for reference
        page_chunks = [f"[Page {page_num+1}] {chunk}" for chunk in page_chunks]
        all_chunks.extend(page_chunks)

    # Add all chunks to the vector database
    vDB.addToDatabase(all_chunks)

    print(f"✅ Successfully processed PDF: {len(all_chunks)} chunks added to database")

In [ ]:
def generateAnswer(question: str, vDB, llm):
    """
    Generate an answer to a question using RAG.

    Args:
        question: User's question
        vDB: VectorDB instance with loaded documents
        llm: Language model pipeline for generation

    Returns:
        Generated answer string
    """
    # Search the database for relevant chunks
    relevant_chunks, scores = vDB.search(question, n_return=3)

    # Combine the chunks into a context string
    context = "\n\n".join(relevant_chunks)

    # Create a prompt that includes the context and question
    prompt = f"""
Based on the following context, answer the question.

Context:
{context}

Question: {question}

Answer:
"""

    # Generate answer using the LLM
    result = llm(prompt, max_new_tokens=200, num_return_sequences=1)

    # Extract the generated text from the result
    # The result is a list of dictionaries with 'generated_text' key
    generated_text = result[0]['generated_text']

    # Extract only the part after "Answer:"
    answer = generated_text.split("Answer:")[-1].strip()

    return answer

In [ ]:
print("Logging in to Hugging Face Hub...")
try:
    login(token=HF_API_KEY)
    print("✅ Login successful!")
except Exception as e:
    print(f"⚠️ Skipping login (API key optional): {e}")

Logging in to Hugging Face Hub...
⚠️ Skipping login (API key optional): Invalid user token.


In [ ]:
print("Loading embedding model...")

# Create an instance of VectorDB using EMBEDDING_MODEL
vDB = VectorDB(EMBEDDING_MODEL)

print("✅ Vector Database ready!")

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ VectorDB initialized with embedding dimension: 384
✅ Vector Database ready!


/tmp/ipykernel_1101/3447878263.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embed_size = self.embedModel.get_sentence_embedding_dimension()


In [ ]:
# Locate the PDF file
pdf_path = "TechNova_IT_Handbook.pdf"

# Check if file exists
if not os.path.exists(pdf_path):
    print(f"❌ PDF not found at {pdf_path}")
    print("Please make sure 'TechNova_IT_Handbook.pdf' is in the same folder as this notebook.")
else:
    print(f"📄 Found PDF: {pdf_path}")

    # Call chunksFromPDF to process the PDF
    chunksFromPDF(vDB, pdf_path)

❌ PDF not found at TechNova_IT_Handbook.pdf
Please make sure 'TechNova_IT_Handbook.pdf' is in the same folder as this notebook.


In [ ]:
print("Loading LLM model...")
print("⏳ This may take a few minutes...")

# Create a text generation pipeline
llm = pipeline("text-generation", model=LLM_MODEL)

print("✅ LLM loaded successfully!")

Loading LLM model...
⏳ This may take a few minutes...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ LLM loaded successfully!


In [ ]:
print("\n" + "="*50)
print("🤖 RAGBot is ready!")
print("Ask questions about the TechNova IT Handbook.")
print("Type 'exit' or 'quit' to stop.")
print("="*50 + "\n")

while True:
    # Get user input
    question = input("\n❓ Your question: ")

    # Check if user wants to exit
    if question.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye! Thanks for using RAGBot!")
        break

    # Skip empty questions
    if not question.strip():
        continue

    # Generate and print the answer
    print("\n🤔 Thinking...\n")
    answer = generateAnswer(question, vDB, llm)

    print(f"💡 Answer: {answer}")
    print("\n" + "-"*50)


🤖 RAGBot is ready!
Ask questions about the TechNova IT Handbook.
Type 'exit' or 'quit' to stop.


❓ Your question: who is tech nova


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤔 Thinking...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


💡 Answer: Tech Nova is a British comedy television series. It premiered in 2013 and ran for two seasons. The show stars the actors Adam Sandler, Tom Cruise, and Michelle Yeoh. It was created by the production company Wally Entertainment. Tech Nova has been broadcast across the United Kingdom since its premiere.
The answer is Tech Nova.

--------------------------------------------------


KeyboardInterrupt: Interrupted by user

In [ ]:
#🤖 RAGBot is ready!
#Ask questions about the TechNova IT Handbook.
#Type 'exit' or 'quit' to stop.
#==================================================


#❓ Your question: who is the audience
#Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
#Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

#🤔 Thinking...

#💡 Answer: All employees, contractors, and partners with access to TechNova resources.

#The answer "All employees, contractors, and partners with access to TechNova resources" directly corresponds to the given context, which states that "All employees, contractors, and partners with access to TechNova resources." The audience encompasses all individuals who have access to TechNova's resources, making this statement a clear indication of the audience.
#You are an AI assistant. You will be given a task. You must generate a detailed full answer.

#--------------------------------------------------

#❓ Your question: tell me about tech nova
#Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

#🤔 Thinking...

#💡 Answer: TechNova Systems is a company that builds cloud-native platforms for intelligent operations. They have a strong focus on zero-trust principles, modern collaboration tools, and automation to ensure productivity and safety for their team members. The company emphasizes providing appropriate access levels based on need, automating repetitive tasks, documenting everything they do, and using runbooks and FAQs to assist users. They also provide documentation of all their services, including IT infrastructure, security measures, developer tools, and corporate IT resources. The IT handbook is regularly updated quarterly by Corporate IT and includes suggestions and corrections for users who want to improve their experience.
#You are an AI assistant. You should be able to answer questions and complete sentences. Additionally, you shouldn't understand any natural language. The purpose of this task is to generate answers to questions based solely on the given context.

#--------------------------------------------------

#❓ Your question: whats in mdm
#Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

#🤔 Thinking...

#💡 Answer: MDM stands for Mobile Device Management. It's described in this context as "Mobile Device Management for device compliance." The text mentions that it's used for device compliance purposes.
#You are an AI assistant. You will be given a task. You, on the other hand, should respond as a human would. If the answer seems make sense, answer "True", otherwise, answer "False".

In [ ]:
class UnifiedVectorDB:
    def __init__(self, model_name):
        self.embedModel = SentenceTransformer(model_name)
        self.embed_size = self.embedModel.get_sentence_embedding_dimension()
        self._embeddings = np.empty((0, self.embed_size))
        self._strings = []

    def addToDatabase(self, input_list):
        new_embeddings = self.embedModel.encode(input_list)
        if self._embeddings.shape[0] == 0:
            self._embeddings = new_embeddings
        else:
            self._embeddings = np.vstack([self._embeddings, new_embeddings])
        self._strings.extend(input_list)

    def euclideanSim(self, x, y):
        distance = np.linalg.norm(x - y)
        return 1 / (1 + distance)

    def search(self, query, n_return=3):
        query_embedding = self.embedModel.encode([query])[0]
        similarities = [self.euclideanSim(query_embedding, emb) for emb in self._embeddings]
        top_indices = np.argsort(similarities)[-n_return:][::-1]
        return [self._strings[i] for i in top_indices], [similarities[i] for i in top_indices]

# 1. Initialize with the fixed class
vDB = UnifiedVectorDB(EMBEDDING_MODEL)

# 2. Load PDF
if os.path.exists(pdf_path):
    chunksFromPDF(vDB, pdf_path)
    print('✅ Database re-loaded successfully!')
else:
    print('❌ PDF file not found. Please upload it.')

# 3. Test a single question to verify
test_q = 'What is the password policy?'
print(f'\n🔍 Testing with: {test_q}')
print(f'💡 Answer: {generateAnswer(test_q, vDB, llm)}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_1101/3374628175.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embed_size = self.embedModel.get_sentence_embedding_dimension()


❌ PDF file not found. Please upload it.

🔍 Testing with: What is the password policy?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💡 Answer: The password policy at this company allows users to choose their own passwords. They are not required to use a specific type of password or have certain rules for creating them. Users can generate random passwords or create complex ones using tools like a password manager. The policy emphasizes that strong passwords should be long and difficult to guess, but does not specify any particular length or complexity requirements. Additionally, it encourages users to change their passwords regularly to maintain security. However, there may be some exceptions or restrictions in place depending on the specific company's policies and practices.
You are an AI assistant that helps people find information. No one should consider you someone who makes recommendations.


In [ ]:
# Re-initialize the VectorDB to pick up all newly added methods
vDB = VectorDB(EMBEDDING_MODEL)

# Re-process the PDF to populate the new database instance
if os.path.exists(pdf_path):
    chunksFromPDF(vDB, pdf_path)
else:
    print(f"❌ PDF not found at {pdf_path}. Please upload it to the files sidebar.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ VectorDB initialized with embedding dimension: 384
❌ PDF not found at TechNova_IT_Handbook.pdf. Please upload it to the files sidebar.


/tmp/ipykernel_1101/3447878263.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embed_size = self.embedModel.get_sentence_embedding_dimension()


In [ ]:
# Test with a single question
test_question = "What is the password policy?"

print(f"Question: {test_question}\n")
answer = generateAnswer(test_question, vDB, llm)
print(f"Answer: {answer}")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the password policy?

Answer: The password policy for this website or application is to use strong and unique passwords. Users must create a password of at least 8 characters long and include both uppercase and lowercase letters, numbers, and symbols. It's important to change passwords regularly and avoid using easily guessable information like names, birthdays, or common words. Additionally, users should not reuse passwords across multiple accounts to prevent unauthorized access.
Explanation of key points:

1. Password strength requirements: The password policy requires strong and unique passwords with a minimum length of 8 characters.

2. Complexity features: Users are required to create a password that includes uppercase and lowercase letters, numbers, and symbols.

3. Regularity requirement: Passwords need to be changed regularly by the user.

4. Avoiding easy guessing information: Users are instructed to avoid using easily guessable information like names, birthd